# **KNN From Scratch with Bayesian Optimization**

**Implementation of K-Nearest Neighbors algorithm from scratch with hyperparameter tuning**

## **Imports**

In [9]:
import pandas as pd
import pickle
from pathlib import Path
import numpy as np
from collections import Counter
from sklearn.metrics import (
    accuracy_score, 
    precision_score, 
    recall_score, 
    f1_score,
    classification_report, 
    confusion_matrix,
    roc_auc_score,
    roc_curve,
    matthews_corrcoef
)
from sklearn.base import BaseEstimator, ClassifierMixin
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

## **KNN Implementation From Scratch**

In [10]:
class KNNFromScratch(BaseEstimator, ClassifierMixin):
    """
    K-Nearest Neighbors Classifier implemented from scratch.
    Compatible with scikit-learn's API for use with BayesSearchCV.
    """
    
    def __init__(self, n_neighbors=5, weights='uniform', metric='euclidean', p=2):
        """
        Initialize KNN classifier.
        
        Parameters:
        -----------
        n_neighbors : int, default=5
            Number of neighbors to use
        weights : str, default='uniform'
            Weight function: 'uniform' or 'distance'
        metric : str, default='euclidean'
            Distance metric: 'euclidean', 'manhattan', or 'minkowski'
        p : int, default=2
            Power parameter for Minkowski distance (1=Manhattan, 2=Euclidean)
        """
        self.n_neighbors = n_neighbors
        self.weights = weights
        self.metric = metric
        self.p = p
        
    def _calculate_distance(self, x1, x2):
        """Calculate distance between two points based on the selected metric."""
        if self.metric == 'euclidean':
            return np.sqrt(np.sum((x1 - x2) ** 2))
        elif self.metric == 'manhattan':
            return np.sum(np.abs(x1 - x2))
        elif self.metric == 'minkowski':
            return np.sum(np.abs(x1 - x2) ** self.p) ** (1 / self.p)
        else:
            raise ValueError(f"Unknown metric: {self.metric}")
    
    def fit(self, X, y):
        """
        Fit the KNN model.
        
        Parameters:
        -----------
        X : array-like of shape (n_samples, n_features)
            Training data
        y : array-like of shape (n_samples,)
            Target values
        """
        # Convert to numpy arrays if needed
        self.X_train_ = np.array(X) if not isinstance(X, np.ndarray) else X
        self.y_train_ = np.array(y) if not isinstance(y, np.ndarray) else y
        self.classes_ = np.unique(self.y_train_)
        return self
    
    def _predict_single(self, x):
        """Predict class for a single sample."""
        # Calculate distances to all training samples
        distances = np.array([self._calculate_distance(x, x_train) 
                             for x_train in self.X_train_])
        
        # Get indices of k nearest neighbors
        k_indices = np.argsort(distances)[:self.n_neighbors]
        k_nearest_labels = self.y_train_[k_indices]
        
        # Apply weighting
        if self.weights == 'uniform':
            # Simple majority vote
            most_common = Counter(k_nearest_labels).most_common(1)
            return most_common[0][0]
        else:  # distance weighting
            k_distances = distances[k_indices]
            # Avoid division by zero
            k_distances = np.where(k_distances == 0, 1e-10, k_distances)
            weights = 1 / k_distances
            
            # Weighted vote
            weighted_votes = {}
            for label, weight in zip(k_nearest_labels, weights):
                weighted_votes[label] = weighted_votes.get(label, 0) + weight
            
            return max(weighted_votes, key=weighted_votes.get)
    
    def predict(self, X):
        """
        Predict class labels for samples in X.
        
        Parameters:
        -----------
        X : array-like of shape (n_samples, n_features)
            Test samples
            
        Returns:
        --------
        y_pred : array of shape (n_samples,)
            Predicted class labels
        """
        X = np.array(X) if not isinstance(X, np.ndarray) else X
        return np.array([self._predict_single(x) for x in X])
    
    def predict_proba(self, X):
        """
        Predict class probabilities for samples in X.
        
        Parameters:
        -----------
        X : array-like of shape (n_samples, n_features)
            Test samples
            
        Returns:
        --------
        proba : array of shape (n_samples, n_classes)
            Class probabilities
        """
        X = np.array(X) if not isinstance(X, np.ndarray) else X
        probas = []
        
        for x in X:
            # Calculate distances to all training samples
            distances = np.array([self._calculate_distance(x, x_train) 
                                 for x_train in self.X_train_])
            
            # Get k nearest neighbors
            k_indices = np.argsort(distances)[:self.n_neighbors]
            k_nearest_labels = self.y_train_[k_indices]
            
            # Calculate probabilities
            if self.weights == 'uniform':
                # Simple count-based probability
                class_proba = np.array([np.sum(k_nearest_labels == c) / self.n_neighbors 
                                       for c in self.classes_])
            else:  # distance weighting
                k_distances = distances[k_indices]
                k_distances = np.where(k_distances == 0, 1e-10, k_distances)
                weights = 1 / k_distances
                
                # Weighted probability
                class_proba = np.zeros(len(self.classes_))
                for i, c in enumerate(self.classes_):
                    mask = k_nearest_labels == c
                    class_proba[i] = np.sum(weights[mask])
                
                # Normalize
                class_proba = class_proba / np.sum(class_proba)
            
            probas.append(class_proba)
        
        return np.array(probas)
    
    def score(self, X, y):
        """Return the mean accuracy on the given test data and labels."""
        return accuracy_score(y, self.predict(X))

print("KNN From Scratch class implementation complete!")

KNN From Scratch class implementation complete!


## **Loading Data**

In [11]:
# Define base path
base_path = Path('../../../../Merged_Data_Preprocessing')

# Load training data (SMOTE-Tomek)
X_train = pd.read_parquet(base_path / 'Normalized_Data_split/X_train_smote_tomek.parquet')
with open(base_path / 'Resampled_Data_split/smote_tomek/y_smote_tomek.pkl', 'rb') as f:
    y_train = pickle.load(f)

# Load test data
X_test = pd.read_parquet(base_path / 'Resampled_Data_split/test/X_test.parquet')
with open(base_path / 'Resampled_Data_split/test/y_test.pkl', 'rb') as f:
    y_test = pickle.load(f)

# Load validation data
X_val = pd.read_parquet(base_path / 'Resampled_Data_split/val/X_val.parquet')
with open(base_path / 'Resampled_Data_split/val/y_val.pkl', 'rb') as f:
    y_val = pickle.load(f)

# Display shapes
print("Training set:")
print(f"  X_train: {X_train.shape}")
print(f"  y_train: {y_train.shape if hasattr(y_train, 'shape') else len(y_train)}")
print("\nTest set:")
print(f"  X_test: {X_test.shape}")
print(f"  y_test: {y_test.shape if hasattr(y_test, 'shape') else len(y_test)}")
print("\nValidation set:")
print(f"  X_val: {X_val.shape}")
print(f"  y_val: {y_val.shape if hasattr(y_val, 'shape') else len(y_val)}")

Training set:
  X_train: (30266, 64)
  y_train: (30266,)

Test set:
  X_test: (3477, 64)
  y_test: (3477,)

Validation set:
  X_val: (3477, 64)
  y_val: (3477,)


## **Initial Model Training**

In [12]:
# Initialize KNN model with default parameters
knn_initial = KNNFromScratch(
    n_neighbors=11,
    weights='distance',
    metric='manhattan',
    p=1
)

# Train the model
print("Training initial KNN model from scratch...")
knn_initial.fit(X_train, y_train)
print("Training complete!")
print(f"Model parameters: n_neighbors={knn_initial.n_neighbors}, weights={knn_initial.weights}, metric={knn_initial.metric}")

Training initial KNN model from scratch...
Training complete!
Model parameters: n_neighbors=11, weights=distance, metric=manhattan


### **Initial Predictions**

In [ ]:
# Make predictions
print("Making predictions on training set...")
y_train_pred_initial = knn_initial.predict(X_train)
print("Making predictions on validation set...")
y_val_pred_initial = knn_initial.predict(X_val)
print("Making predictions on test set...")
y_test_pred_initial = knn_initial.predict(X_test)

# Get probability predictions
print("Calculating probability predictions...")
y_train_proba_initial = knn_initial.predict_proba(X_train)[:, 1]
y_val_proba_initial = knn_initial.predict_proba(X_val)[:, 1]
y_test_proba_initial = knn_initial.predict_proba(X_test)[:, 1]

print("Predictions complete!")

Making predictions on training set...
Making predictions on validation set...
Making predictions on test set...
Calculating probability predictions...


### **Evaluation Function**

In [ ]:
def evaluate_model(y_true, y_pred, y_proba, dataset_name):
    """Evaluate model performance and display metrics"""
    print(f"\n{'='*60}")
    print(f"{dataset_name} SET EVALUATION")
    print(f"{'='*60}")
    
    # Calculate confusion matrix for specificity
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    
    # Calculate metrics
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)  # Sensitivity/TPR
    specificity = tn / (tn + fp)  # TNR
    f1 = f1_score(y_true, y_pred)
    mcc = matthews_corrcoef(y_true, y_pred)
    roc_auc = roc_auc_score(y_true, y_proba)
    
    print(f"\nAccuracy:    {accuracy:.4f}")
    print(f"Precision:   {precision:.4f}")
    print(f"Recall:      {recall:.4f}")
    print(f"Specificity: {specificity:.4f}")
    print(f"F1-Score:    {f1:.4f}")
    print(f"MCC:         {mcc:.4f}")
    print(f"ROC-AUC:     {roc_auc:.4f}")
    
    print(f"\n{'-'*60}")
    print("Classification Report:")
    print(f"{'-'*60}")
    print(classification_report(y_true, y_pred, target_names=['No Fire', 'Fire']))
    
    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'specificity': specificity,
        'f1_score': f1,
        'mcc': mcc,
        'roc_auc': roc_auc
    }

print("Evaluation function defined.")

Evaluation function defined.


### **Initial Model Evaluation**

In [ ]:
# Evaluate initial model on all datasets
train_metrics_initial = evaluate_model(y_train, y_train_pred_initial, y_train_proba_initial, "TRAINING (INITIAL)")
val_metrics_initial = evaluate_model(y_val, y_val_pred_initial, y_val_proba_initial, "VALIDATION (INITIAL)")
test_metrics_initial = evaluate_model(y_test, y_test_pred_initial, y_test_proba_initial, "TEST (INITIAL)")


TRAINING (INITIAL) SET EVALUATION

Accuracy:    1.0000
Precision:   1.0000
Recall:      1.0000
Specificity: 1.0000
F1-Score:    1.0000
MCC:         1.0000
ROC-AUC:     1.0000

------------------------------------------------------------
Classification Report:
------------------------------------------------------------
              precision    recall  f1-score   support

     No Fire       1.00      1.00      1.00     15133
        Fire       1.00      1.00      1.00     15133

    accuracy                           1.00     30266
   macro avg       1.00      1.00      1.00     30266
weighted avg       1.00      1.00      1.00     30266


VALIDATION (INITIAL) SET EVALUATION

Accuracy:    0.8873
Precision:   0.3047
Recall:      0.7413
Specificity: 0.8962
F1-Score:    0.4319
MCC:         0.4280
ROC-AUC:     0.8852

------------------------------------------------------------
Classification Report:
------------------------------------------------------------
              precision    

In [ ]:
# Visualizations
from sklearn.metrics import precision_recall_curve, average_precision_score

def plot_confusion(y_true, y_pred, title):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(4.5, 4))
    sns.heatmap(
        cm,
        annot=True,
        fmt='d',
        cmap='Blues',
        cbar=False,
        xticklabels=['No Fire', 'Fire'],
        yticklabels=['No Fire', 'Fire'],
    )
    plt.title(f"Confusion Matrix - {title}")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.tight_layout()
    plt.show()

def plot_roc_pr(y_true, y_proba, title):
    fpr, tpr, _ = roc_curve(y_true, y_proba)
    precision, recall, _ = precision_recall_curve(y_true, y_proba)
    roc_auc = roc_auc_score(y_true, y_proba)
    ap = average_precision_score(y_true, y_proba)

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].plot(fpr, tpr, label=f"AUC = {roc_auc:.3f}")
    axes[0].plot([0, 1], [0, 1], linestyle='--', color='gray', linewidth=1)
    axes[0].set_title(f"ROC Curve - {title}")
    axes[0].set_xlabel("False Positive Rate")
    axes[0].set_ylabel("True Positive Rate")
    axes[0].legend(loc="lower right")
    axes[0].grid(alpha=0.3)

    axes[1].plot(recall, precision, label=f"AP = {ap:.3f}")
    axes[1].set_title(f"Precision-Recall - {title}")
    axes[1].set_xlabel("Recall")
    axes[1].set_ylabel("Precision")
    axes[1].legend(loc="lower left")
    axes[1].grid(alpha=0.3)

    plt.tight_layout()
    plt.show()

# Confusion matrices
plot_confusion(y_train, y_train_pred_initial, "Training")
plot_confusion(y_val, y_val_pred_initial, "Validation")
plot_confusion(y_test, y_test_pred_initial, "Test")

# ROC + PR curves
plot_roc_pr(y_train, y_train_proba_initial, "Training")
plot_roc_pr(y_val, y_val_proba_initial, "Validation")
plot_roc_pr(y_test, y_test_proba_initial, "Test")

#save the images
output_path = Path('../../../../../Rapport_TP/images/KNN')
plt.savefig(output_path / 'confusion_matrix_training_initial_from_scratch.png')

NameError: name 'y_train' is not defined